In [1]:
import subprocess, sys, os

env = os.environ.copy()
env["START_YEAR"] = "2021"
env["END_YEAR"] = "2026"
env["NETWORK_MODE"] = "global"

result = subprocess.run(
    [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
)

print(result.stdout)
print(result.stderr)


C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026-06-10 19:14:02,822 [INFO] ============================================================
2026-06-10 19:14:02,822 [INFO] CWTS Export — Edge Weight Builder
2026-06-10 19:14:02,823 [INFO]   Mode: GLOBAL  |  Years: 2021-2026
2026-06-10 19:14:02,823 [INFO]   Edge weights: ON
2026-06-10 19:14:02,823 [INFO] ============================================================
2026-06-10 19:14:02,823 [INFO] Fetching top 5 Frontiers journals by publication count...
c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\squirrel\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-06-10 19:14:08,286 [INFO]   Journal IDs: [3315714752512, 2405181685761, 29

In [2]:
import subprocess

result = subprocess.run(
    [
        "java",
        "-cp",
        "publicationclassification.jar",
        "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)

PublicationClassificationCreator version 1.1.0
By Nees Jan van Eck
Centre for Science and Technology Studies (CWTS), Leiden University

Usage: PublicationClassificationCreator
	<pub_file> <cit_link_file> <classification_file>
	<largest_component> <n_iterations>
	<resolution_micro_level> <pub_threshold_micro_level>
	<resolution_meso_level> <pub_threshold_meso_level>
	<resolution_macro_level> <pub_threshold_macro_level>
		(to create a publication classification based on data in text files)

   or  PublicationClassificationCreator
	<server> <database> <pub_table> <cit_link_table> <classification_table>
	<largest_component> <n_iterations>
	<resolution_micro_level> <pub_threshold_micro_level>
	<resolution_meso_level> <pub_threshold_meso_level>
	<resolution_macro_level> <pub_threshold_macro_level>
		(to create a publication classification based on data in an SQL Server database)

Arguments:
<pub_file>
	Name of the publications input file. This text file must contain two tab-separated
	column

In [3]:
result = subprocess.run(
    [
        "java",
        "-cp",
        "publicationclassification.jar",
        "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
        "cwts_output/pubs.txt",
        "cwts_output/cit_links.txt",
        "cwts_output/classification.txt",
        "true",  # largest component only
        "100",  # iterations
        "2e-4", #micro
        "2000",  # micro
        "5e-5", #meso
        "10000",  # meso
        "3e-7", # macro
        "20000",  # macro
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

PublicationClassificationCreator version 1.1.0
By Nees Jan van Eck
Centre for Science and Technology Studies (CWTS), Leiden University

Reading citation network from file... 
Exception in thread "main" java.lang.OutOfMemoryError: Java heap space
	at java.base/java.util.Arrays.copyOf(Arrays.java:3681)
	at nl.cwts.util.LargeDoubleArray.ensureCapacity(LargeDoubleArray.java:421)
	at nl.cwts.util.LargeDoubleArray.append(LargeDoubleArray.java:332)
	at nl.cwts.publicationclassification.run.FileIO.readNetwork(FileIO.java:129)
	at nl.cwts.publicationclassification.run.PublicationClassificationCreator.main(PublicationClassificationCreator.java:255)



In [4]:
import pandas as pd
df = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)

print(f"Total classified: {len(df):,}")
for level in ["micro", "meso", "macro"]:
    vc = df[level].value_counts()
    print(f"\n{level.upper()}: {len(vc):,} clusters")
    print(f"  Largest : {vc.iloc[0]:,} ({vc.iloc[0]/len(df)*100:.1f}%)")
    print(f"  Smallest: {vc.iloc[-1]:,}")
    print(f"  Median  : {vc.median():.0f}")

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Total classified: 2,443,009

MICRO: 640 clusters
  Largest : 24,710 (1.0%)
  Smallest: 2,003
  Median  : 3408

MESO: 132 clusters
  Largest : 53,649 (2.2%)
  Smallest: 10,109
  Median  : 16634

MACRO: 7 clusters
  Largest : 743,630 (30.4%)
  Smallest: 20,720
  Median  : 309759


### Labelling with GPT

In [5]:
import label_clusters

# Run the script
label_clusters.main()

Loading classification...
Loading titles...


ParserError: Error tokenizing data. C error: Expected 5 fields in line 645685, saw 7


In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("cwts_output")
classif_path = OUTPUT_DIR / "classification.txt"
titles_path = OUTPUT_DIR / "pub_titles.txt"
classif = pd.read_csv(
    classif_path,
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)

print("Loading titles...")
titles = pd.read_csv(
    titles_path,
    sep="\t",
    header=None,
    names=["pub_no", "title"],
)

merged = classif.merge(titles, on="pub_no")
print(f"Merged: {len(merged):,} publications")

# merged.sample(5000).to_csv('full_merged.csv')

Loading titles...
Merged: 2,415,653 publications


In [ ]:
merged.sample(5000).to_csv("merged_sample.csv")

In [ ]:
pd.set_option("display.max_colwidth", None)
merged[['macro','title']].sample(500)

,macro,title
2193685,0,OBEClust: joining forces to prevent obesity
1671552,1,Bacterial species-structure-property relationships of polyhydroxyalkanoate biopolymers produced on simple sugars for thin film applications
44835,1,Innovative integrated carbon paper supported Bi/rGO/MnO2 cathode with dual redox reactions for aqueous zinc-ion batteries
1540896,3,Learning Natural Categories: Effects of Interleaving Practice in Children and Young Adults
2063894,2,MPC with adaptive-bias RBFNN and Q-reinforcement learning for fault-tolerant control of nonlinear processes
...,...,...
182252,1,High efficiency green and red emitting CdSeTe quantum dots: Synthesis and PLQY improvement for optical applications
1055711,0,The association between frailty and hospital-related adverse events in older hospitalised patients: a systematic literature review
719709,3,Blended learning in higher education for the development of intrinsic motivation: a systematic review
146099,0,Systematic Review and Meta-analysis of Outcomes of the Advanta V12 or iCAST Bridging Stent Graft Used for Fenestrated and Branched Endovascular Aortic Repair


### looking at scope

In [ ]:
import pandas as pd
from pathlib import Path

core = pd.read_csv(
    "cwts_output/frontiers_core.txt",
    sep="\t",
    header=None,
    names=["pub_no", "is_frontiers", "journal"],
)

print(core["journal"].unique())

['Aids Reviews'
 'Journal of the National Institute for Career Education and Counselling'
 'Revista Latinoamericana de Ciencias Sociales Niñez y Juventud' ...
 'Cairn.info' 'Obogashchenie Rud' '«GEOTAR-Media» Publishing Group eBooks']


In [ ]:
import importlib
import journal_scope

# Override config variables
journal_scope.SCOPE_LEVEL = "macro"
journal_scope.SCOPE_THRESHOLD = 0.80
journal_scope.MIN_PAPERS = 50
journal_scope.USE_GPT = False
journal_scope.OUTPUT_DIR = Path("cwts_output")

journal_scope.TARGET_JOURNALS = [
    "Frontiers in Immunology",
    "Frontiers in Public Health",
    "Frontiers in Medicine",
    "Frontiers in Oncology",
    "Frontiers in Psychology",
]

# Run
journal_scope.main()

Loading classification...
Loading titles...
Loading journal assignments...
Merged: 14,184 publications across 5 journals
After MIN_PAPERS=50 filter: 5 journals, 14,184 papers

Computing scope at 'macro' level (threshold=80%)...
  Frontiers in Immunology                  n= 4,650  core_clusters=  1  OOS=13.1%
  Frontiers in Medicine                    n= 2,344  core_clusters=  2  OOS=16.6%
  Frontiers in Oncology                    n= 2,163  core_clusters=  1  OOS=19.4%
  Frontiers in Psychology                  n= 2,177  core_clusters=  2  OOS=18.7%
  Frontiers in Public Health               n= 2,850  core_clusters=  3  OOS=13.8%

Generating GPT scope descriptions...
  Frontiers in Immunology: This journal publishes research on various aspects of immunology, including the ...
  Frontiers in Medicine: This journal publishes research on a wide range of medical topics, including adv...
  Frontiers in Oncology: This journal publishes research on various aspects of oncology, including clini

In [ ]:
pd.set_option("display.max_colwidth", 50)
pd.read_csv('cwts_output/journal_scope.csv')

,journal,n_papers,n_core_clusters,core_clusters,n_in_scope,n_oos,oos_rate,top5_clusters,scope_level,scope_threshold,scope_statement,core_topics
0,Frontiers in Oncology,2163,1,[0],1743,420,0.1942,"0(1743), 2(214), 1(95), 4(44), 3(33)",macro,0.8,This journal publishes research on various asp...,"Cancer treatment, Molecular mechanisms, Risk m..."
1,Frontiers in Psychology,2177,2,"[0, 3]",1769,408,0.1874,"3(1293), 0(476), 2(279), 1(51), 4(50)",macro,0.8,This journal publishes research on various asp...,"Mental Health, Emotional Well-being, Social Be..."
2,Frontiers in Medicine,2344,2,"[0, 2]",1955,389,0.1660,"0(1742), 2(213), 3(199), 1(99), 4(48)",macro,0.8,This journal publishes research on a wide rang...,"Machine Learning in Medicine, Clinical Outcome..."
3,Frontiers in Public Health,2850,3,"[0, 2, 3]",2457,393,0.1379,"0(1291), 3(837), 2(329), 4(177), 6(111)",macro,0.8,This journal publishes research on a wide rang...,"Public Health, Mental Health, Disease Preventi..."
4,Frontiers in Immunology,4650,1,[0],4040,610,0.1312,"0(4040), 1(202), 2(161), 4(100), 6(89)",macro,0.8,This journal publishes research on various asp...,"Immunotherapy, Autoimmunity, Cancer Immunology..."
